# Component 3 — Counterfactual Group-Membership Fairness (Scenario: Spurious Observed Disparity, No Retained Dependence)

Scenario:
- Race and gender correlate with covariates (structural imbalance).
- The model uses covariates; group membership does not add predictive signal once covariates are known.
- Component 1 may show observed disparities; Component 3 should yield small u-values (near 0).

In [ ]:
import numpy as np
import pandas as pd

from lightgbm import LGBMClassifier
!pip install --force-reinstall --no-deps --no-cache-dir git+https://github.com/vsubbian/FairLogue.git

import FairLogue

ModuleNotFoundError: No module named 'lightgbm'

In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def make_synth_c3_no_retained_dependence(n=10000, seed=31):
    rng = np.random.default_rng(seed)

    gender_factor = rng.integers(0, 2, size=n)
    race = rng.choice(["White", "Black or African American"], size=n, p=[0.7, 0.3])
    race_factor = (race == "Black or African American").astype(int)

    # Structural imbalance: covariates differ by group
    # Example: Black patients have slightly higher comorbidity prevalence due to correlated features.
    base_age = rng.normal(60, 12, size=n)
    age = (base_age + 2.0 * race_factor + 0.5 * gender_factor).clip(18, 90)

    htn = rng.binomial(1, sigmoid((age - 55) / 10 + 0.35 * race_factor), size=n)
    diabetes = rng.binomial(1, 0.16 + 0.12 * (htn == 1) + 0.05 * race_factor, size=n)
    chf = rng.binomial(1, sigmoid((age - 66) / 10 + 0.25 * race_factor), size=n)
    stroke_prior = rng.binomial(1, 0.06 + 0.06 * (chf == 1), size=n)
    age_75 = (age >= 75).astype(int)

    # Outcome depends ONLY on covariates (no direct group effect)
    logit = (
        -3.2
        + 0.045 * (age - 50)
        + 0.95 * chf
        + 0.55 * htn
        + 0.70 * diabetes
        + 0.90 * stroke_prior
        + 0.20 * age_75
    )
    p = sigmoid(logit)
    target = rng.binomial(1, p, size=n)

    df = pd.DataFrame({
        "gender_factor": gender_factor.astype(int),
        "race_factor": race_factor.astype(int),
        "age": age,
        "chf": chf,
        "htn": htn,
        "age_75": age_75,
        "diabetes": diabetes,
        "stroke_prior": stroke_prior,
        "target": target.astype(int),
    })
    return df

df3 = make_synth_c3_no_retained_dependence()
df3.head()

In [ ]:
features = ["age", "gender_factor", "race_factor", "chf", "htn", "age_75", "diabetes", "stroke_prior"]

MODEL_PARAMS = {
    "objective": "binary",
    "n_estimators": 550,
    "num_leaves": 31,
    "learning_rate": 0.05,
    "random_state": 42,
    "class_weight": "balanced",
}

lgbm = LGBMClassifier(**MODEL_PARAMS)

m3 = FairLogue.Component3.Model(
    data=df3,
    outcome="target",
    protected_characteristics=("race_factor", "gender_factor"),
    covariates=features,
    outcome_estimator=lgbm,
    method="sr",
    n_splits=5,
    random_state=42,
)

m3.pre_process_data()
m3.fit_fairness(
    cutoff=None,
    gen_null=True,
    bootstrap="rescaled",
)

summary3 = m3.summarize()
summary3

In [ ]:
est3, delta3, uvals3 = m3.plots(delta_uval=0.10)
(est3, delta3, uvals3)